# Phase 1.5: Posture Merge

Merge posture xlsx (RULA + QEC) into the cleaned survey by **severity rank**: sort survey by an exposure-severity score (pain + fatigue + hours), sort posture by RULA Table C, pair rank-to-rank.

## Step 0 — Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

CWD = Path.cwd()
ROOT = CWD if (CWD / 'data').exists() else CWD.parent
CLEANED = ROOT / 'data' / 'processed' / 'cleaned.csv'
POSTURE = ROOT / 'data' / 'raw' / 'posture_data.xlsx'
OUT     = ROOT / 'data' / 'processed'

## Step 1 — Load survey and posture

In [2]:
survey = pd.read_csv(CLEANED)
print('survey:', survey.shape)

posture = pd.read_excel(POSTURE)
# Drop the empty spacer column
posture = posture.drop(columns=[c for c in posture.columns if c.startswith('Unnamed')], errors='ignore')
print('posture:', posture.shape)
posture.head()

survey: (182, 48)


posture: (182, 9)


,IMAGE,TABLE A SCORE,TABLE B SCORE,TABLE C / Score,BACK,SHOULDER/ARM,WRIST/HAND,NECK,STRESS
0,NaN,3,4,4,24,30,32,14.0,16
1,NaN,4,4,6,18,20,30,10.0,16
2,NaN,5,4,8,22,22,30,14.0,16
3,NaN,3,5,6,14,20,26,10.0,16
4,NaN,1,4,4,18,20,20,8.0,16


## Step 2 — Compute survey exposure severity (pain + fatigue + hours)

In [3]:
# Pain count: number of 'Yes' across the 9 Nordic body-area items (last 12 months)
nordic_cols = [c for c in survey.columns if 'In the past 12 months' in c]
pain_count = (survey[nordic_cols] == 'Yes').sum(axis=1)

# Fatigue: mean of the 6 Borg CR10 items (last 6 columns, already 0-10)
borg_cols = list(survey.columns[-6:])
fatigue_mean = survey[borg_cols].mean(axis=1)

# Working hours per day -> midpoint number
hours_map = {'< 3 hrs': 2, '3-5 hrs': 4, '6-8 hrs': 7, '> 8 hrs': 9}
hours_num = survey['Working Hours per Day'].map(hours_map)

# Normalise each component to 0-1 so they contribute equally to the rank
exposure_severity = (
    pain_count   / pain_count.max()
  + fatigue_mean / fatigue_mean.max()
  + hours_num    / hours_num.max()
)

survey['pain_count']        = pain_count
survey['fatigue_mean']      = fatigue_mean.round(2)
survey['hours_num']         = hours_num
survey['exposure_severity'] = exposure_severity.round(3)

survey[['pain_count', 'fatigue_mean', 'hours_num', 'exposure_severity']].describe()

,pain_count,fatigue_mean,hours_num,exposure_severity
count,182.000000,182.000000,182.000000,182.000000
mean,3.956044,4.530549,7.346154,1.759181
std,2.772844,1.938360,1.965437,0.517550
min,0.000000,0.000000,2.000000,0.463000
25%,2.000000,3.170000,7.000000,1.411750
50%,4.000000,4.415000,7.000000,1.731500
75%,6.000000,5.957500,9.000000,2.106500
max,9.000000,9.000000,9.000000,2.963000


## Step 3 — Compute posture severity (RULA Table C)

In [4]:
# RULA Table C is the final RULA grand score (1-7+). Higher = worse posture.
posture = posture.rename(columns={'TABLE C / Score': 'rula_table_c',
                                  'TABLE A SCORE':   'rula_table_a',
                                  'TABLE B SCORE':   'rula_table_b',
                                  'BACK':         'qec_back',
                                  'SHOULDER/ARM': 'qec_shoulder_arm',
                                  'WRIST/HAND':   'qec_wrist_hand',
                                  'NECK':         'qec_neck',
                                  'STRESS':       'qec_stress'})

posture['posture_severity'] = posture['rula_table_c']

print('posture rows:', len(posture))
posture[['rula_table_a', 'rula_table_b', 'rula_table_c', 'qec_back',
         'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck', 'qec_stress']].describe()

posture rows: 182


,rula_table_a,rula_table_b,rula_table_c,qec_back,qec_shoulder_arm,qec_wrist_hand,qec_neck,qec_stress
count,182.000000,182.000000,182.000000,182.000000,182.000000,182.000000,181.000000,182.000000
mean,3.796703,3.505495,5.681319,23.857143,27.494505,29.637363,14.011050,8.510989
std,0.533300,1.233722,1.096243,5.372472,6.787010,7.449300,2.516587,5.027811
min,1.000000,2.000000,3.000000,8.000000,10.000000,10.000000,8.000000,1.000000
25%,4.000000,2.250000,5.000000,20.000000,22.000000,26.000000,12.000000,4.000000
50%,4.000000,3.000000,6.000000,24.000000,28.000000,30.000000,14.000000,9.000000
75%,4.000000,4.000000,6.000000,26.000000,32.000000,32.000000,16.000000,15.000000
max,5.000000,7.000000,8.000000,48.000000,56.000000,46.000000,20.000000,16.000000


## Step 4 — Sort both by severity and pair rank-to-rank

In [5]:
assert len(survey) == len(posture), f'survey={len(survey)} vs posture={len(posture)}'

# Survey ranked by exposure severity (descending: highest = rank 1)
survey_sorted = survey.sort_values('exposure_severity', ascending=False).reset_index(drop=True)
survey_sorted['rank'] = survey_sorted.index + 1

# Posture ranked by posture severity (descending)
posture_sorted = posture.sort_values('posture_severity', ascending=False).reset_index(drop=True)
posture_sorted['rank'] = posture_sorted.index + 1

# Merge on rank
posture_cols = ['rula_table_a', 'rula_table_b', 'rula_table_c',
                'qec_back', 'qec_shoulder_arm', 'qec_wrist_hand', 'qec_neck', 'qec_stress',
                'posture_severity', 'rank']
merged = survey_sorted.merge(posture_sorted[posture_cols], on='rank', suffixes=('', '_p'))

print('merged shape:', merged.shape)
merged[['exposure_severity', 'rank', 'rula_table_c', 'qec_back', 'qec_neck']].head(10)

merged shape: (182, 62)


,exposure_severity,rank,rula_table_c,qec_back,qec_neck
0,2.963,1,8,22,14.0
1,2.944,2,7,20,14.0
2,2.926,3,7,26,16.0
3,2.907,4,7,24,14.0
4,2.889,5,7,26,16.0
5,2.870,6,7,20,16.0
6,2.833,7,7,20,16.0
7,2.815,8,7,34,16.0
8,2.815,9,7,24,16.0
9,2.778,10,7,24,14.0


## Step 5 — Save

In [6]:
out = OUT / 'cleaned_with_posture.csv'
merged.to_csv(out, index=False)
print('saved', out, merged.shape)

saved F:\Ergonomic_Project\data\processed\cleaned_with_posture.csv (182, 62)
